In [0]:
%sql
create schema if not exists weather.silver;

In [0]:
source_table= "weather.bronze.current_weather"
target_table= "weather.silver.current_weather"

In [0]:
from pyspark.sql.functions import col
from pyspark.sql import functions as f

bronze_df = spark.read.table(source_table)

silver_df = (
    bronze_df.select(
        col("city"),
        # Current weather
        col("data.current.temperature").alias("temperature"),
        col("data.current.feelslike").alias("feels_like"),
        col("data.current.humidity").alias("humidity"),
        col("data.current.pressure").alias("pressure"),
        col("data.current.visibility").alias("visibility"),
        col("data.current.wind_speed").alias("wind_speed"),
        col("data.current.wind_dir").alias("wind_direction"),
        # Air Quality
        col("data.current.air_quality.co").cast("double").alias("co"),
        col("data.current.air_quality.no2").cast("double").alias("no2"),
        col("data.current.air_quality.o3").cast("double").alias("o3"),
        col("data.current.air_quality.pm10").cast("double").alias("pm10"),
        col("data.current.air_quality.pm2_5").cast("double").alias("pm25"),
        # Location
        col("data.location.country").alias("country"),
        col("data.location.region").alias("region"),
        col("data.location.lat").cast("double").alias("latitude"),
        col("data.location.lon").cast("double").alias("longitude"),
    )
    .withColumn("load_dt_tm", f.current_timestamp())
    .withColumn("update_dt_tm", f.current_timestamp())
)
display(silver_df)

In [0]:
(silver_df.write.mode("overwrite").format("delta").saveAsTable(target_table))

In [0]:
%sql
select * from weather.silver.current_weather